## Bronze (Raw data ingestion)

**Step 1: Initialize Logging and Import Spark Delta Modules**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("abcgroup")


**Step 2: Load Utility Functions for ETL Pipeline Execution**

In [0]:
%run /Users/tumisomphahle14@gmail.com/etl_pipeline/script/common/utilities

**Step 3: Set Widget Defaults for Catalog Volume and Sources**

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")
dbutils.widgets.text("volume", "raw_sources", "Volume")
dbutils.widgets.text("data_source_one", "source_crm", "Data Source One")
dbutils.widgets.text("data_source_two", "source_erp", "Data Source Two")

**Step 4: Set Widget Defaults for Catalog Volume and Sources**

In [0]:
catalog = dbutils.widgets.get("catalog")
volume = dbutils.widgets.get("volume")
data_source_one = dbutils.widgets.get("data_source_one")
data_source_two = dbutils.widgets.get("data_source_two")

**Step 5: Configure Initial Data Sources and File Paths for Ingestion**

In [0]:
INITIAL_INGESTION = [
    {
        "source": "crm",
        "path": f"/Volumes/{catalog}/{bronze_schema}/{volume}/{data_source_one}/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": f"/Volumes/{catalog}/{bronze_schema}/{volume}/{data_source_one}/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": f"/Volumes/{catalog}/{bronze_schema}/{volume}/{data_source_one}/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": f"/Volumes/{catalog}/{bronze_schema}/{volume}/{data_source_two}/CUST_AZ12.csv",
        "table": "erp_cust_az12"
    },
    {
        "source": "erp",
        "path": f"/Volumes/{catalog}/{bronze_schema}/{volume}/{data_source_two}/LOC_A101.csv",
        "table": "erp_loc_a101"
    },
    {
        "source": "erp",
        "path": f"/Volumes/{catalog}/{bronze_schema}/{volume}/{data_source_two}/PX_CAT_G1V2.csv",
        "table": "erp_px_cat_g1v2"
    }
]

**Step 6: Ingest CSV Files into Bronze Schema with Spark Delta**

In [0]:
for item in INITIAL_INGESTION:
    logger.info(f"Ingesting {item['source']} data into {catalog} {bronze_schema} schema as table {item['table']}")
    df = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(item["path"])
    )
    (
        df.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(f"{catalog}.{bronze_schema}.{item['table']}")
    )
    logger.info(f"Successfully ingested data into {catalog} {bronze_schema} schema as table {item['table']}")
